In [ ]:
import pandas as pd
import numpy as np
import jieba
import jieba.analyse
from gensim import corpora, models
import matplotlib.pyplot as plt
import sys

In [ ]:
gbk = False 
path = '/kaggle/input/ssr-paper/data with label.xlsx'
if gbk:
        data = pd.read_excel(path,encoding='gbk') 
else:
        data = pd.read_excel(path)

In [ ]:
data.head(3)

# Word Clouds

In [ ]:
# import pandas as pd
# import re

# def clean_text(text):
#     # ???? <br> ??(??????)
#     cleaned_text = re.sub(r'<br>', '', text, flags=re.IGNORECASE)
#     return cleaned_text

# def extract_info(text, keywords):
#     # ?????????,?????????
#     pattern = '|'.join(map(re.escape, keywords))
#     pattern = r'(' + pattern + r'):'
#     matches = list(re.finditer(pattern, text))
    
#     # ???????
#     matches.sort(key=lambda x: x.start())
    
#     result = {}
#     if not matches:
#         return result
    
#     for i in range(len(matches)):
#         keyword = matches[i].group(1)
#         start = matches[i].end()  # ?????????
        
#         if i < len(matches) - 1:
#             next_start = matches[i+1].start()
#             value = text[start:next_start].strip()
#         else:
#             value = text[start:].strip()
            
#         result[keyword] = value
    
#     return result

# keywords = [
#     "??????",
#     "????",
#     "????",
#     "????",
#     "????",
#     "????",
#     "????",
#     "????",
# ]

# # ?????????
# data['cleaned_text'] = data['??'].apply(clean_text)
# data['extracted_info'] = data['cleaned_text'].apply(lambda x: extract_info(x, keywords))

# # ??????????
# extracted_df = pd.json_normalize(data['extracted_info'])
# data = pd.concat([data, extracted_df], axis=1)

# # ???????
# data.drop(columns=['cleaned_text', 'extracted_info'], inplace=True)

# data

In [ ]:
# from wordcloud import WordCloud
# import matplotlib.pyplot as plt
# import jieba

# def generate_chinese_wordcloud(data_column, stopwords=None):
#     # ???????????????
#     text = ' '.join(data_column.dropna())
    
#     # ??jieba??????
#     words = jieba.lcut(text)
    
#     # ???????????
#     text_for_wordcloud = ' '.join(words)
    
#     # ?????????(?????)
#     if stopwords is None:
#         stopwords = set()
#     else:
#         stopwords = set(stopwords)
    
#     # ?????????
#     default_stopwords = {
#         '?', '?', '?', '?', '?', '?', '?', '?', '?', '?', '?', '?', '??', '?', '?', '?', '?', '?', '?', '?', '?', '?', '?', '??', '?', '?', '??', '?', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '???', '??', '???', '?', '?', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '??', '????', '??', '??', '??', '??', '????', '?', '?', '?', '?', '??', '??', '??', '??'
#     }
    
#     # ???????
#     stopwords.update(default_stopwords)
    
#     # ????
#     wordcloud = WordCloud(
#         width=800,
#         height=400,
#         background_color='white',
#         font_path='/kaggle/input/cn-fonts/simhei.ttf',  # ????????
#         stopwords=stopwords
#     ).generate(text_for_wordcloud)
    
#     # ?????
#     plt.figure(figsize=(10, 5))
#     plt.imshow(wordcloud, interpolation='bilinear')
#     plt.axis('off')  # ?????
#     plt.show()

In [ ]:
# custom_stopwords = {'??','??','r'}
# generate_chinese_wordcloud(data['??????'], custom_stopwords)

In [ ]:
# custom_stopwords = {'??','??','r','??','??'}
# generate_chinese_wordcloud(data['????'], custom_stopwords)

In [ ]:
# custom_stopwords = {'??','??','r','????'}
# generate_chinese_wordcloud(data['????'], custom_stopwords)

In [ ]:
# custom_stopwords = {'??','??','r'}
# generate_chinese_wordcloud(data['????'], custom_stopwords)

In [ ]:
# custom_stopwords = {'??','??','r','?','?','???','?','?','?','?','??','??'}
# generate_chinese_wordcloud(data['????'], custom_stopwords)

In [ ]:
# custom_stopwords = {'??','??','r'}
# generate_chinese_wordcloud(data['????'], custom_stopwords)

In [ ]:
# custom_stopwords = {'??','??','r'}
# generate_chinese_wordcloud(data['????'], custom_stopwords)

In [ ]:
# custom_stopwords = {'??','??','r'}
# generate_chinese_wordcloud(data['????'], custom_stopwords)

# Word Segmentation and Cleansing

In [ ]:
def load_stopwords(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        stopwords = set(line.strip() for line in f if line.strip())
        stopwords.add('br')
    return stopwords
def segment_and_remove_stopwords(text, stopwords):
    words = jieba.lcut(text) 
    filtered_words = [word for word in words if word not in stopwords and word.strip()]  
    return filtered_words

In [ ]:
stopwords_file = '/kaggle/input/chinese-stopwords/stopwords.txt' 
stopwords = load_stopwords(stopwords_file)
data['???'] = data['??'].apply(lambda x: segment_and_remove_stopwords(x, stopwords))

In [ ]:
data[['???','label']].head(3)

# BERT-TextCNN

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import BertModel, BertTokenizer

class BERT_TextCNN(nn.Module):
    def __init__(self, bert_path, num_filters=100, kernel_sizes=[3,4,5], num_classes=4):
        super(BERT_TextCNN, self).__init__()
        self.bert = BertModel.from_pretrained(bert_path)
        self.hidden_size = self.bert.config.hidden_size

        # TextCNN??
        self.convs = nn.ModuleList([
            nn.Conv2d(1, num_filters, (k, self.hidden_size)) for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), num_classes)

    def forward(self, input_ids, attention_mask):
        # BERT??
        bert_output = self.bert(input_ids, attention_mask=attention_mask)
        last_hidden_state = bert_output.last_hidden_state  # (batch_size, seq_len, hidden_size)
        
        # ??????????
        x = last_hidden_state.unsqueeze(1)  # ?????? -> (batch, 1, seq_len, hidden_size)
        
        # ???????
        conv_outputs = []
        for conv in self.convs:
            # ????
            conv_out = conv(x)          # (batch, num_filters, seq_len - k +1, 1)
            conv_out = F.relu(conv_out) # ????
            # ????
            pooled = F.max_pool1d(
                conv_out.squeeze(3),    # ????????
                conv_out.size(2)        # ??????????????
            ).squeeze(2)                # ????????
            conv_outputs.append(pooled)
        
        # ??????????
        concat = torch.cat(conv_outputs, dim=1)
        concat = self.dropout(concat)
        logits = self.fc(concat)
        
        return logits

In [ ]:
import torch
import pandas as pd
from torch.utils.data import TensorDataset, DataLoader
from transformers import BertTokenizer
from sklearn.model_selection import train_test_split  # ????????

texts = data['???'].tolist() 
original_labels = data['label'].tolist() 
texts = [" ".join(text) for text in texts] 

train_texts, test_texts, train_labels, test_labels = train_test_split(
    texts, original_labels, test_size=0.2, random_state=42)

train_labels = [label - 1 for label in train_labels]
test_labels = [label - 1 for label in test_labels]

In [ ]:
print(len(train_labels))
print(len(test_labels))

## Embedding

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-Chinese')

In [ ]:
def preprocess(texts, max_length=512):
    inputs = tokenizer(
        texts,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors='pt'
    )
    return inputs['input_ids'], inputs['attention_mask']

train_input_ids, train_attention_mask = preprocess(train_texts)
test_input_ids, test_attention_mask = preprocess(test_texts)
train_labels = torch.as_tensor(train_labels, dtype=torch.long)
test_labels = torch.as_tensor(test_labels, dtype=torch.long)

model = BERT_TextCNN(bert_path='bert-base-uncased', num_classes=4)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)

train_dataset = TensorDataset(
    train_input_ids, 
    train_attention_mask,  
    train_labels
)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

## Training

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ?????????
train_losses = []

# ???????????(??????)
plt.ion()
fig, ax = plt.subplots(figsize=(10, 6))

for epoch in range(10): 
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    # ????epoch?????
    avg_loss = total_loss / len(train_loader)
    train_losses.append(avg_loss)
    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}")
    
    # ??epoch?????
    ax.clear()  # ????????
    ax.plot(np.arange(1, len(train_losses)+1), train_losses, marker='o', linestyle='-', color='b')
    ax.set_title('Training Loss Progress')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.grid(True)
    plt.pause(0.1)  # ?????
    plt.draw()

# ??????????????
plt.ioff()  # ??????
plt.figure(figsize=(10, 6))
plt.plot(np.arange(1, len(train_losses)+1), train_losses, marker='o', linestyle='-', color='r')
plt.title('Training Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.savefig('training_loss.png')  # ???????
plt.show()  # ??????

# ????????
torch.save(model.state_dict(), "final_model.pt")
torch.cuda.empty_cache()  # ????

## Evaluation

In [ ]:
from torch.utils.data import DataLoader
test_dataset = TensorDataset(
    test_input_ids, 
    test_attention_mask, 
    test_labels
)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)  # ?????? shuffle

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

def evaluate_and_plot(model, data_loader, title_prefix, device):
    model.eval()
    all_predictions = []
    all_true_labels = []
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids, attention_mask, labels = [b.to(device) for b in batch]
            logits = model(input_ids, attention_mask)
            predictions = torch.argmax(logits, dim=1).cpu().numpy()
            all_predictions.extend(predictions)
            all_true_labels.extend(labels.cpu().numpy())
    
    true_labels = np.array(all_true_labels)
    predictions = np.array(all_predictions)
    
    # ????
    accuracy = (predictions == true_labels).mean()
    report = classification_report(true_labels, predictions, output_dict=True)
    precision = report['weighted avg']['precision']
    recall = report['weighted avg']['recall']
    f1 = report['weighted avg']['f1-score']
    
    print(f"{title_prefix} Set Results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}\n")
    
    # ??????????
    cm_normalized = confusion_matrix(
        true_labels, 
        predictions, 
        normalize='true'  # ?????,?????1
    )
    
    # ??????
    cm_normalized = np.round(cm_normalized, decimals=2)
    
    # ?????
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm_normalized, 
        annot=True, 
        fmt=".2f",  # ??????
        cmap="Blues",
        xticklabels=['Class 0', 'Class 1', 'Class 2', 'Class 3'],
        yticklabels=['Class 0', 'Class 1', 'Class 2', 'Class 3']
    )
    plt.title(f'{title_prefix} Set Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()
    
    return true_labels, predictions

true, pred = evaluate_and_plot(model, test_loader, "Test", device)

In [ ]:
df = pd.DataFrame({'True_Labels': true + 1, 'Predicted_Labels': pred + 1})
df.to_csv("test_results.csv",index=False)